# PIDA-RL Diabetes - 06. Entrenamiento y evaluacion de DQN

Este notebook entrena y evalua un agente Deep Q-Network (DQN) sobre el mismo simulador utilizado para las politicas base. Es autocontenido: solo requiere la cohorte persistente de Google Drive.

> Los resultados son experimentos sobre un entorno simulado. No constituyen evidencia clinica ni recomendaciones de atencion.


## 1. Inicializacion

El notebook detecta Drive, recupera el repositorio temporal si es necesario y usa la cohorte canonica persistente.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json

PROJECT_NAME = 'pida-rl-datos-publicos'
REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
PROJECT_ROOT = Path('/content') / PROJECT_NAME

DRIVE_ROOT = next(
    (p for p in [Path('/content/drive'), Path('/content/gdrive')] if (p / 'MyDrive').is_dir()),
    None
)

if DRIVE_ROOT is None:
    from google.colab import drive
    mount_point = Path('/content/drive')
    if mount_point.exists() and any(mount_point.iterdir()):
        mount_point = Path('/content/gdrive')
    if mount_point.exists() and any(mount_point.iterdir()):
        raise RuntimeError('No hay un punto de montaje vacio para Google Drive. Reinicia la sesion.')
    drive.mount(str(mount_point), force_remount=False)
    DRIVE_ROOT = mount_point

MYDRIVE_ROOT = DRIVE_ROOT / 'MyDrive'
PERSISTENT_ROOT = MYDRIVE_ROOT / 'PIDA-RL-Diabetes'
DATA_ROOT = PERSISTENT_ROOT / 'data'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
MODELS_DIR = PERSISTENT_ROOT / 'models'
TABLES_DIR = PERSISTENT_ROOT / 'results' / 'tables'
LOGS_DIR = PERSISTENT_ROOT / 'logs'

for directory in [MODELS_DIR, TABLES_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(PROJECT_ROOT)])
elif not (PROJECT_ROOT / '.git').is_dir():
    raise RuntimeError(f'La ruta no es un repositorio Git: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)

COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'
BASELINES_PATH = TABLES_DIR / 'evaluacion_politicas_base_v1.csv'

if not COHORT_PATH.is_file():
    raise FileNotFoundError(f'No existe la cohorte: {COHORT_PATH}. Ejecuta primero el notebook 03.')

print('Proyecto temporal:', PROJECT_ROOT)
print('Cohorte persistente:', COHORT_PATH)
print('Resultados de referencia disponibles:', BASELINES_PATH.is_file())


## 2. Cargar y validar la cohorte


In [ ]:
import numpy as np
import pandas as pd

df_cohorte_diabetes = pd.read_csv(COHORT_PATH, dtype={'FOLIO_INT': 'string'})

required_columns = [
    'FOLIO_INT', 'edad', 'sexo', 'anios_con_diabetes',
    'insulina_diaria', 'depresion', 'consultas_control_12m',
    'num_hospitalizaciones', 'num_complicaciones',
    'score_riesgo', 'categoria_riesgo'
]

missing = [column for column in required_columns if column not in df_cohorte_diabetes.columns]
if missing:
    raise KeyError('Faltan columnas en cohorte: ' + ', '.join(missing))

numeric_columns = [
    'edad', 'sexo', 'anios_con_diabetes', 'insulina_diaria',
    'depresion', 'consultas_control_12m', 'num_hospitalizaciones',
    'num_complicaciones', 'score_riesgo'
]

assert np.isfinite(df_cohorte_diabetes[numeric_columns].to_numpy(dtype=float)).all()
assert df_cohorte_diabetes['categoria_riesgo'].isin(['bajo', 'medio', 'alto']).all()
assert not df_cohorte_diabetes['FOLIO_INT'].duplicated().any()

print('Cohorte valida:', df_cohorte_diabetes.shape)
display(df_cohorte_diabetes['categoria_riesgo'].value_counts().rename_axis('riesgo').reset_index(name='n'))


## 3. Definir el entorno autocontenido

Esta definicion es intencionalmente identica a la utilizada para las politicas base. Asi se evita depender de archivos temporales de sesiones anteriores y se preserva la comparabilidad entre politicas.


In [ ]:
try:
    import gymnasium as gym
    from gymnasium import spaces
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'gymnasium'])
    import gymnasium as gym
    from gymnasium import spaces

class DiabetesFollowUpEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(self, cohorte_df, capacidad_mensual=0.20, horizonte=12, seed=None):
        super().__init__()
        required = [
            'edad', 'sexo', 'anios_con_diabetes', 'insulina_diaria',
            'depresion', 'consultas_control_12m', 'num_hospitalizaciones',
            'num_complicaciones', 'score_riesgo', 'categoria_riesgo'
        ]
        missing = [name for name in required if name not in cohorte_df.columns]
        if missing:
            raise KeyError('Faltan columnas: ' + ', '.join(missing))
        if not 0 < float(capacidad_mensual) <= 1:
            raise ValueError('capacidad_mensual debe estar entre 0 y 1.')
        if int(horizonte) < 1:
            raise ValueError('horizonte debe ser al menos 1.')
        self.cohorte_base = cohorte_df.reset_index(drop=True).copy()
        self.n_perfiles = len(self.cohorte_base)
        if self.n_perfiles == 0:
            raise ValueError('La cohorte no puede estar vacia.')
        numeric = [
            'edad', 'sexo', 'anios_con_diabetes', 'insulina_diaria',
            'depresion', 'consultas_control_12m', 'num_hospitalizaciones',
            'num_complicaciones', 'score_riesgo'
        ]
        if not np.isfinite(self.cohorte_base[numeric].to_numpy(dtype=float)).all():
            raise ValueError('La cohorte contiene NaN o infinitos.')
        if not self.cohorte_base['categoria_riesgo'].isin({'bajo', 'medio', 'alto'}).all():
            raise ValueError('categoria_riesgo invalida.')
        self.capacidad_mensual = float(capacidad_mensual)
        self.horizonte = int(horizonte)
        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(13,), dtype=np.float32)
        self.costo_accion = {0: 0, 1: 1, 2: 2, 3: 3}
        self.reduccion_evento = {0: 0.00, 1: 0.20, 2: 0.45, 3: 0.65}
        self.riesgo_ordinal = {'bajo': 0, 'medio': 1, 'alto': 2}
        self.reset(seed=seed)

    @staticmethod
    def _norm(value, denominator):
        return float(np.clip(float(value) / max(float(denominator), 1.0), 0.0, 1.0))

    def _risk_level(self, category):
        return self.riesgo_ordinal[str(category)]

    def _get_obs(self, idx):
        row = self.cohorte.iloc[idx]
        remaining = self.presupuesto_mensual - self.recursos_usados_mes
        sex_norm = 0.0 if float(row['sexo']) == 1.0 else 1.0
        risk_norm = self._risk_level(row['categoria_riesgo']) / 2.0
        return np.array([
            self._norm(row['edad'], 100),
            sex_norm,
            self._norm(row['anios_con_diabetes'], 80),
            float(np.clip(row['insulina_diaria'], 0, 1)),
            float(np.clip(row['depresion'], 0, 1)),
            self._norm(row['consultas_control_12m'], 24),
            self._norm(row['num_hospitalizaciones'], 12),
            self._norm(row['num_complicaciones'], 5),
            self._norm(row['score_riesgo'], 16),
            risk_norm,
            self._norm(row['meses_sin_contacto'], self.horizonte),
            self._norm(self.mes_actual, self.horizonte),
            self._norm(remaining, self.presupuesto_mensual)
        ], dtype=np.float32)

    def _event_probability(self, row, risk_level):
        value = (
            0.01 + 0.025 * risk_level
            + 0.010 * float(row['num_complicaciones'])
            + 0.020 * float(row['num_hospitalizaciones'] > 0)
            + 0.005 * min(float(row['meses_sin_contacto']), 6)
        )
        return float(np.clip(value, 0.0, 0.80))

    def _reward(self, action, risk_level, action_accepted, adverse_event, over_capacity):
        reward = 0.0
        if action_accepted and action > 0:
            reward += {1: 0.5, 2: 1.0, 3: 1.5}[action] * (risk_level + 1)
        if action == 0 or not action_accepted:
            reward -= {0: 0.0, 1: 0.5, 2: 1.5}[risk_level]
        if adverse_event:
            reward -= 5.0
        if over_capacity:
            reward -= 3.0
        return float(reward)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.cohorte = self.cohorte_base.copy()
        self.cohorte['meses_sin_contacto'] = 0
        self.mes_actual = 1
        self.idx_actual = 0
        self.presupuesto_mensual = max(1, int(round(self.n_perfiles * self.capacidad_mensual)))
        self.recursos_usados_mes = 0
        self.contactos_aceptados_mes = 0
        self.intentos_exceso_mes = 0
        obs = self._get_obs(self.idx_actual)
        info = {
            'mes': self.mes_actual,
            'indice_perfil': self.idx_actual,
            'n_perfiles': self.n_perfiles,
            'presupuesto_mensual': self.presupuesto_mensual,
            'recursos_usados_mes': 0,
            'recursos_restantes_mes': self.presupuesto_mensual
        }
        return obs, info

    def step(self, action):
        action = int(action)
        if not self.action_space.contains(action):
            raise ValueError(f'Accion invalida: {action}')
        idx = self.idx_actual
        month = self.mes_actual
        row = self.cohorte.iloc[idx]
        risk_level = self._risk_level(row['categoria_riesgo'])
        requested_cost = self.costo_accion[action]
        over_capacity = self.recursos_usados_mes + requested_cost > self.presupuesto_mensual
        action_accepted = not over_capacity
        actual_cost = requested_cost if action_accepted else 0
        if over_capacity:
            self.intentos_exceso_mes += 1
        else:
            self.recursos_usados_mes += actual_cost
            if action > 0:
                self.contactos_aceptados_mes += 1
        base_probability = self._event_probability(row, risk_level)
        reduction = self.reduccion_evento[action] if action_accepted else 0.0
        probability = float(np.clip(base_probability * (1.0 - reduction), 0.0, 1.0))
        adverse_event = bool(self.np_random.random() < probability)
        reward = self._reward(action, risk_level, action_accepted, adverse_event, over_capacity)
        if action > 0 and action_accepted:
            self.cohorte.loc[idx, 'meses_sin_contacto'] = 0
        else:
            self.cohorte.loc[idx, 'meses_sin_contacto'] += 1
        used = int(self.recursos_usados_mes)
        remaining = int(self.presupuesto_mensual - self.recursos_usados_mes)
        contacts = int(self.contactos_aceptados_mes)
        excess_attempts = int(self.intentos_exceso_mes)
        self.idx_actual += 1
        terminated = False
        truncated = False
        month_changed = False
        if self.idx_actual >= self.n_perfiles:
            self.idx_actual = 0
            self.mes_actual += 1
            month_changed = True
            self.recursos_usados_mes = 0
            self.contactos_aceptados_mes = 0
            self.intentos_exceso_mes = 0
            if self.mes_actual > self.horizonte:
                terminated = True
        obs = np.zeros(self.observation_space.shape, dtype=np.float32) if terminated else self._get_obs(self.idx_actual)
        info = {
            'mes': month,
            'mes_siguiente': self.mes_actual,
            'cambio_mes': month_changed,
            'indice_perfil': idx,
            'nivel_riesgo': risk_level,
            'categoria_riesgo': str(row['categoria_riesgo']),
            'evento_adverso': adverse_event,
            'prob_evento': probability,
            'excede_capacidad': over_capacity,
            'accion_aceptada': action_accepted,
            'costo_solicitado': requested_cost,
            'costo_real': actual_cost,
            'presupuesto_mensual': self.presupuesto_mensual,
            'recursos_usados_mes': used,
            'recursos_restantes_mes': remaining,
            'contactos_aceptados_mes': contacts,
            'intentos_exceso_mes': excess_attempts
        }
        return obs, reward, terminated, truncated, info

print('Entorno DQN definido.')


## 4. Validacion antes del entrenamiento

Esta validacion debe terminar sin excepciones antes de entrenar. `VecCheckNan` se usara tambien durante el entrenamiento para detectar valores invalidos.


In [ ]:
try:
    import stable_baselines3
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'stable-baselines3'])

from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecCheckNan

env_check = DiabetesFollowUpEnv(df_cohorte_diabetes, capacidad_mensual=0.20, horizonte=12, seed=42)
check_env(env_check, warn=True)

obs, info = env_check.reset(seed=42)
for step in range(250):
    action = env_check.action_space.sample()
    obs, reward, terminated, truncated, info = env_check.step(action)
    assert np.isfinite(obs).all(), f'Observacion no finita en paso {step}'
    assert np.isfinite(reward), f'Recompensa no finita en paso {step}'
    if terminated or truncated:
        break

print('Entorno validado para DQN.')
print('Espacio:', env_check.observation_space, env_check.action_space)


## 5. Crear DQN

Se inicia con una corrida diagnostica de 10000 pasos. Si termina correctamente, se continua hasta completar 100000 pasos sin reiniciar el contador.


In [ ]:
from stable_baselines3 import DQN

def make_env(seed):
    def _init():
        return DiabetesFollowUpEnv(
            cohorte_df=df_cohorte_diabetes,
            capacidad_mensual=0.20,
            horizonte=12,
            seed=seed
        )
    return _init

env_dqn = DummyVecEnv([make_env(42)])
env_dqn = VecCheckNan(env_dqn, raise_exception=True)

DQN_MODEL_PATH = MODELS_DIR / 'dqn_diabetes_followup_v1'

model_dqn = DQN(
    policy='MlpPolicy',
    env=env_dqn,
    learning_rate=3e-4,
    buffer_size=100000,
    learning_starts=5000,
    batch_size=128,
    gamma=0.95,
    train_freq=4,
    gradient_steps=1,
    target_update_interval=1000,
    exploration_fraction=0.50,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.05,
    policy_kwargs={'net_arch': [128, 128]},
    tensorboard_log=str(LOGS_DIR / 'dqn_v1'),
    verbose=1,
    seed=42,
    device='auto'
)

print('Modelo DQN creado.')
print('Ruta del modelo:', str(DQN_MODEL_PATH) + '.zip')


## 6. Diagnostico inicial de acciones

El modelo sin entrenar puede seleccionar acciones arbitrarias. El diagnostico posterior al entrenamiento es obligatorio para detectar una politica degenerada que elija siempre no contacto.


In [ ]:
def diagnose_actions(model, cohort, n_steps=2000, seed=42):
    env = DiabetesFollowUpEnv(cohort, capacidad_mensual=0.20, horizonte=12, seed=seed)
    obs, _ = env.reset(seed=seed)
    rows = []

    for step in range(n_steps):
        action, _ = model.predict(obs, deterministic=True)
        action = int(action)
        next_obs, reward, terminated, truncated, info = env.step(action)
        rows.append({
            'etapa': 'diagnostico',
            'paso': step + 1,
            'accion': action,
            'categoria_riesgo': info['categoria_riesgo'],
            'accion_aceptada': info['accion_aceptada'],
            'recompensa': reward
        })
        obs = next_obs
        if terminated or truncated:
            break

    return pd.DataFrame(rows)

df_diagnostic_before = diagnose_actions(model_dqn, df_cohorte_diabetes, n_steps=2000, seed=42)
display(df_diagnostic_before['accion'].value_counts().sort_index().rename_axis('accion').reset_index(name='frecuencia'))


## 7. Entrenamiento diagnostico


In [ ]:
DIAGNOSTIC_TIMESTEPS = 10000

model_dqn.learn(total_timesteps=DIAGNOSTIC_TIMESTEPS, progress_bar=True)

print('Entrenamiento diagnostico terminado.')


## 8. Revisar politica tras la corrida diagnostica

No avances a la corrida completa si la celda arroja NaN o si el modelo no puede producir acciones validas. Una politica con 100% accion 0 se conserva como hallazgo experimental, pero debe documentarse.


In [ ]:
df_diagnostic_after = diagnose_actions(model_dqn, df_cohorte_diabetes, n_steps=5000, seed=42)

action_distribution_after = (
    df_diagnostic_after['accion']
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .rename_axis('accion')
    .reset_index(name='porcentaje')
)

display(action_distribution_after)

positive_actions = int((df_diagnostic_after['accion'] > 0).sum())
print('Acciones positivas en diagnostico:', positive_actions)

if positive_actions == 0:
    print('ADVERTENCIA: DQN eligio solo accion 0 en esta corrida diagnostica.')
    print('No es necesario borrar el experimento; se registrara como resultado negativo si persiste.')
else:
    print('DQN eligio al menos una accion de contacto. Puede continuar la corrida completa.')


## 9. Entrenamiento completo

Ejecuta esta celda despues de revisar el diagnostico. El entrenamiento continua desde los 10000 pasos iniciales y completa 100000 pasos totales.


In [ ]:
TOTAL_TIMESTEPS = 100000
REMAINING_TIMESTEPS = TOTAL_TIMESTEPS - DIAGNOSTIC_TIMESTEPS

if REMAINING_TIMESTEPS > 0:
    model_dqn.learn(
        total_timesteps=REMAINING_TIMESTEPS,
        reset_num_timesteps=False,
        progress_bar=True
    )

model_dqn.save(str(DQN_MODEL_PATH))

saved_model = Path(str(DQN_MODEL_PATH) + '.zip')
assert saved_model.is_file()
print('Modelo DQN guardado:', saved_model)
print('Bytes:', saved_model.stat().st_size)


## 10. Evaluar DQN con el protocolo comun

La evaluacion usa tres semillas y reporta resultados por nivel de riesgo.


In [ ]:
def evaluate_dqn(model, cohort, seeds=(42, 123, 2026)):
    summary_rows = []
    step_rows = []

    for seed in seeds:
        env = DiabetesFollowUpEnv(cohort, capacidad_mensual=0.20, horizonte=12, seed=seed)
        obs, _ = env.reset(seed=seed)
        total_reward = 0.0
        episode_rows = []
        step_number = 0

        while True:
            action, _ = model.predict(obs, deterministic=True)
            action = int(action)
            next_obs, reward, terminated, truncated, info = env.step(action)
            step_number += 1
            total_reward += float(reward)

            row = {
                'politica': 'dqn_v1',
                'semilla': int(seed),
                'paso': int(step_number),
                'mes': int(info['mes']),
                'nivel_riesgo': int(info['nivel_riesgo']),
                'categoria_riesgo': info['categoria_riesgo'],
                'accion': action,
                'accion_aceptada': bool(info['accion_aceptada']),
                'contacto_aceptado': bool(action > 0 and info['accion_aceptada']),
                'evento_adverso': bool(info['evento_adverso']),
                'recompensa': float(reward),
                'costo_real': int(info['costo_real']),
                'excede_capacidad': bool(info['excede_capacidad'])
            }

            episode_rows.append(row)
            step_rows.append(row)
            obs = next_obs

            if terminated or truncated:
                break

        episode = pd.DataFrame(episode_rows)
        grouped = episode.groupby(['nivel_riesgo', 'categoria_riesgo'], as_index=False).agg(
            decisiones=('accion', 'size'),
            contactos_aceptados=('contacto_aceptado', 'sum'),
            eventos_adversos=('evento_adverso', 'sum'),
            recursos_usados=('costo_real', 'sum'),
            excesos_capacidad=('excede_capacidad', 'sum')
        )

        grouped['cobertura_porcentaje'] = np.where(
            grouped['decisiones'] > 0,
            100.0 * grouped['contactos_aceptados'] / grouped['decisiones'],
            0.0
        )

        grouped['politica'] = 'dqn_v1'
        grouped['semilla'] = int(seed)
        grouped['recompensa_acumulada_episodio'] = float(total_reward)
        grouped['pasos_episodio'] = int(step_number)
        summary_rows.extend(grouped.to_dict(orient='records'))

    return pd.DataFrame(summary_rows), pd.DataFrame(step_rows)

df_dqn_summary, df_dqn_steps = evaluate_dqn(model_dqn, df_cohorte_diabetes)
display(df_dqn_summary.sort_values(['semilla', 'nivel_riesgo']))


## 11. Diagnostico final, comparacion y persistencia


In [ ]:
df_diagnostic_final = diagnose_actions(model_dqn, df_cohorte_diabetes, n_steps=5000, seed=42)

df_dqn_action_distribution = (
    df_diagnostic_final
    .groupby(['categoria_riesgo', 'accion'], as_index=False)
    .size()
    .rename(columns={'size': 'frecuencia'})
)

df_dqn_action_distribution['porcentaje_en_riesgo'] = (
    100.0 * df_dqn_action_distribution['frecuencia']
    / df_dqn_action_distribution.groupby('categoria_riesgo')['frecuencia'].transform('sum')
)

display(df_dqn_action_distribution.sort_values(['categoria_riesgo', 'accion']))

DQN_SUMMARY_PATH = TABLES_DIR / 'evaluacion_dqn_v1.csv'
DQN_STEPS_PATH = TABLES_DIR / 'evaluacion_pasos_dqn_v1.csv'
DQN_DIAGNOSTIC_PATH = TABLES_DIR / 'diagnostico_acciones_dqn_v1.csv'

df_dqn_summary.to_csv(DQN_SUMMARY_PATH, index=False, encoding='utf-8-sig')
df_dqn_steps.to_csv(DQN_STEPS_PATH, index=False, encoding='utf-8-sig')
df_dqn_action_distribution.to_csv(DQN_DIAGNOSTIC_PATH, index=False, encoding='utf-8-sig')

if BASELINES_PATH.is_file():
    df_baselines = pd.read_csv(BASELINES_PATH)
    df_comparison = pd.concat([df_baselines, df_dqn_summary], ignore_index=True)
    comparison_path = TABLES_DIR / 'comparacion_base_vs_dqn_v1.csv'
    df_comparison.to_csv(comparison_path, index=False, encoding='utf-8-sig')
    print('Comparacion guardada:', comparison_path)
else:
    print('No se encontro evaluacion base; se guardaron solo resultados DQN.')

for path in [DQN_SUMMARY_PATH, DQN_STEPS_PATH, DQN_DIAGNOSTIC_PATH]:
    print(path.name, '| existe:', path.is_file(), '| bytes:', path.stat().st_size if path.is_file() else 0)

assert np.isfinite(df_dqn_steps['recompensa']).all()
assert (df_dqn_steps['costo_real'] >= 0).all()

print('DQN entrenado, evaluado y guardado.')


## Resultado esperado

Deben existir el modelo `dqn_diabetes_followup_v1.zip`, el diagnostico de acciones y los CSV de evaluacion en Drive. Si el DQN adopta una politica degenerada de no contacto, conserva los archivos: ese comportamiento se analizara formalmente en la etapa de resultados y limitaciones.
